# Practical 05 – Multiple and Polynomial Regression

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University


This practical covers categorical encoding, multiple regression, evaluation, and polynomial regression.

In [ ]:
from pathlib import Path
import pandas as pd

def load_course_csv(filename):
    candidates = [Path("data") / filename, Path("Lecture05_Multiple_and_Polynomial_Regression") / "data" / filename]
    local_path = next((p for p in candidates if p.exists()), None)
    url = f"https://raw.githubusercontent.com/asabbah44/ML6402-Machine-Learning-Practicals/main/Lecture05_Multiple_and_Polynomial_Regression/data/{filename}"
    source = local_path if local_path else url
    print("Loaded from:", source)
    return pd.read_csv(source)

In [ ]:
df = load_course_csv("50_AdAgency.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=["Profit"])
y = df["Profit"]

preprocess = ColumnTransformer([
    ("city", OneHotEncoder(drop="first", handle_unknown="ignore"), ["City"])
], remainder="passthrough")

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R²  :", r2_score(y_test, pred))

## Polynomial regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

salary_df = load_course_csv("Position_Salaries.csv")
X_poly = salary_df[["Rank"]]
y_poly = salary_df["Salary"]

linear = LinearRegression().fit(X_poly, y_poly)
poly = make_pipeline(PolynomialFeatures(degree=4, include_bias=False), LinearRegression())
poly.fit(X_poly, y_poly)

print("Linear R²:", linear.score(X_poly, y_poly))
print("Degree-4 polynomial R²:", poly.score(X_poly, y_poly))

grid = pd.DataFrame({"Rank": np.linspace(X_poly["Rank"].min(), X_poly["Rank"].max(), 200)})
plt.scatter(X_poly["Rank"], y_poly)
plt.plot(grid["Rank"], poly.predict(grid))
plt.xlabel("Rank")
plt.ylabel("Salary")
plt.show()

## Exercise
Compare polynomial degrees 2, 3, 4, and 5. Explain why increasing degree can eventually overfit.